# 1. LLM Continuous Batching：怎样让短请求完成后立刻补入新请求？

## 面试回答主线

静态 batching 在一批请求开始后锁住所有 sequence slot，短回答提前结束也要等最长回答完成才允许下一批进入。Continuous batching 在每个 decode step 重新检查完成请求、回收 KV slot，并从等待队列补入新请求，因此主要改善排队延迟与 slot 利用率。调度器至少要记录 arrival、remaining tokens、active set、完成时刻和公平性，而不是只调用一个现成 serving 框架。面试时我会在同一组真实请求上逐 step 模拟静态批与连续批，比较平均延迟、完成时刻和浪费 slot。只追求高优先级或短任务可能让老请求饥饿，所以还需要 aging 或最大等待时间。生产系统还要把 prefill、decode、KV block、adapter 与显存水位纳入准入合同。

## 1. 真实案例：八个错峰到达、回答长度不同的 LLM 请求

每条请求包含自然语言问题、到达 step、prompt token 数、预计生成 token 数和优先级。生成长度用小整数缩放真实工作量，使完整调度轨迹可以直接读懂；batch capacity 固定为 3。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示调度输入与轨迹
requests = [{"id": "Q01", "question": "总结退款政策", "arrival": 0, "prompt_tokens": 120, "new_tokens": 6, "priority": 2}, {"id": "Q02", "question": "订单 A17 到哪了", "arrival": 0, "prompt_tokens": 42, "new_tokens": 2, "priority": 3}, {"id": "Q03", "question": "解释 BM25 参数", "arrival": 0, "prompt_tokens": 86, "new_tokens": 4, "priority": 2}, {"id": "Q04", "question": "生成会议行动项", "arrival": 1, "prompt_tokens": 150, "new_tokens": 3, "priority": 1}, {"id": "Q05", "question": "判断是否需要人工升级", "arrival": 2, "prompt_tokens": 55, "new_tokens": 1, "priority": 4}, {"id": "Q06", "question": "比较 LoRA 与 DoRA", "arrival": 3, "prompt_tokens": 210, "new_tokens": 5, "priority": 2}, {"id": "Q07", "question": "提取发票抬头", "arrival": 4, "prompt_tokens": 35, "new_tokens": 2, "priority": 3}, {"id": "Q08", "question": "给出数据库排障步骤", "arrival": 5, "prompt_tokens": 175, "new_tokens": 4, "priority": 2}]  # 定义八个真实语义且到达时间不同的请求
capacity = 3  # 设置 GPU decode 批次最多容纳三个活动序列
preview = [{"请求": item["id"], "问题": item["question"], "到达step": item["arrival"], "prompt_tokens": item["prompt_tokens"], "生成step": item["new_tokens"], "优先级": item["priority"]} for item in requests]  # 汇总调度器需要观察的业务字段
print("Continuous Batching 请求预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示八条请求的到达与工作量差异

Continuous Batching 请求预览：
[{'请求': 'Q01',
  '问题': '总结退款政策',
  '到达step': 0,
  'prompt_tokens': 120,
  '生成step': 6,
  '优先级': 2},
 {'请求': 'Q02',
  '问题': '订单 A17 到哪了',
  '到达step': 0,
  'prompt_tokens': 42,
  '生成step': 2,
  '优先级': 3},
 {'请求': 'Q03',
  '问题': '解释 BM25 参数',
  '到达step': 0,
  'prompt_tokens': 86,
  '生成step': 4,
  '优先级': 2},
 {'请求': 'Q04',
  '问题': '生成会议行动项',
  '到达step': 1,
  'prompt_tokens': 150,
  '生成step': 3,
  '优先级': 1},
 {'请求': 'Q05',
  '问题': '判断是否需要人工升级',
  '到达step': 2,
  'prompt_tokens': 55,
  '生成step': 1,
  '优先级': 4},
 {'请求': 'Q06',
  '问题': '比较 LoRA 与 DoRA',
  '到达step': 3,
  'prompt_tokens': 210,
  '生成step': 5,
  '优先级': 2},
 {'请求': 'Q07',
  '问题': '提取发票抬头',
  '到达step': 4,
  'prompt_tokens': 35,
  '生成step': 2,
  '优先级': 3},
 {'请求': 'Q08',
  '问题': '给出数据库排障步骤',
  '到达step': 5,
  'prompt_tokens': 175,
  '生成step': 4,
  '优先级': 2}]


## 2. Baseline（基线）：静态 batch 锁住 slot 直到最长序列结束

静态调度只有当前 batch 全部结束后才准入下一批。短序列虽然已经完成，其 slot 仍被当前 batch 保留，下面逐 step 记录 active、真正产生 token 的请求和浪费 slot。

In [2]:
def simulate_static(items, batch_capacity):  # 模拟批次边界不可动态改变的静态调度器
    time = 0  # 初始化离散 decode 时间
    waiting = []  # 初始化已经到达但尚未进入 batch 的请求队列
    active = []  # 初始化当前被静态 batch 锁定的请求槽位
    completion = {}  # 保存每个请求真正生成完最后 token 的时刻
    timeline = []  # 保存每个 decode step 的可解释调度轨迹
    reserved_slots = 0  # 统计 batch 生命周期内总预留 sequence slot
    useful_tokens = 0  # 统计真正生成的 token 数
    while len(completion) < len(items):  # 持续运行直到所有真实请求完成
        waiting.extend(dict(item, remaining=item["new_tokens"]) for item in items if item["arrival"] == time)  # 把当前时刻到达的请求加入等待队列
        if not active and waiting:  # 只有旧 batch 全部退出后才能创建新 batch
            active = waiting[:batch_capacity]  # 按到达顺序拿取最多三个请求
            waiting = waiting[batch_capacity:]  # 从等待队列移除已经准入的请求
        active_before = [slot["id"] for slot in active]  # 记录本 step 被 batch 锁住的全部 slot
        emitted = []  # 收集本 step 真正生成 token 的请求
        if active:  # 当前存在静态 batch 时推进一次 decode
            reserved_slots += len(active)  # 所有 batch 成员无论完成与否都继续占用 slot
            for slot in active:  # 遍历固定 batch 中的每个请求
                if slot["remaining"] > 0:  # 只有尚未完成的序列可以产生新 token
                    slot["remaining"] -= 1  # 消耗当前请求一个生成 step
                    useful_tokens += 1  # 累加实际完成的 token 工作量
                    emitted.append(slot["id"])  # 记录本 step 对哪个请求有用
                    if slot["remaining"] == 0:  # 检查当前 token 是否完成整条回答
                        completion[slot["id"]] = time + 1  # 保存请求完成时刻供延迟计算
            if all(slot["remaining"] == 0 for slot in active):  # 只有整批全部结束才能释放 batch
                active = []  # 释放静态 batch 中包括早已完成的全部 slot
        timeline.append({"step": time, "batch_slots": active_before, "产生token": emitted, "等待": [item["id"] for item in waiting]})  # 保存当前 step 的完整账本
        time += 1  # 推进到下一个 decode step
    return completion, timeline, reserved_slots, useful_tokens  # 返回完成时刻、轨迹与利用率计数
static_completion, static_timeline, static_reserved, static_useful = simulate_static(requests, capacity)  # 在八个请求上运行静态 batching 基线
static_rows = [{"请求": item["id"], "完成step": static_completion[item["id"]], "端到端延迟": static_completion[item["id"]] - item["arrival"]} for item in requests]  # 计算逐请求静态排队加生成延迟
print("静态 batch 前十个 step：")  # 输出基线轨迹标题
pprint(static_timeline[:10], sort_dicts=False)  # 展示短请求完成后 slot 仍不能补入新任务
print("静态 batch 完成结果：")  # 输出基线逐请求结果标题
pprint(static_rows, sort_dicts=False)  # 展示八条请求的完成时刻与延迟

静态 batch 前十个 step：
[{'step': 0,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01', 'Q02', 'Q03'],
  '等待': []},
 {'step': 1,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01', 'Q02', 'Q03'],
  '等待': ['Q04']},
 {'step': 2,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01', 'Q03'],
  '等待': ['Q04', 'Q05']},
 {'step': 3,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01', 'Q03'],
  '等待': ['Q04', 'Q05', 'Q06']},
 {'step': 4,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01'],
  '等待': ['Q04', 'Q05', 'Q06', 'Q07']},
 {'step': 5,
  'batch_slots': ['Q01', 'Q02', 'Q03'],
  '产生token': ['Q01'],
  '等待': ['Q04', 'Q05', 'Q06', 'Q07', 'Q08']},
 {'step': 6,
  'batch_slots': ['Q04', 'Q05', 'Q06'],
  '产生token': ['Q04', 'Q05', 'Q06'],
  '等待': ['Q07', 'Q08']},
 {'step': 7,
  'batch_slots': ['Q04', 'Q05', 'Q06'],
  '产生token': ['Q04', 'Q06'],
  '等待': ['Q07', 'Q08']},
 {'step': 8,
  'batch_slots': ['Q04', 'Q05', 'Q06'],
  '产生token': ['Q04', 'Q06'],
  '等待': ['Q07

## 3. 手写核心机制：每个 decode step 回收并补充 active set

连续调度在 step 开始时吸收到达请求，填满空 slot；生成一个 token 后立即移除完成序列，下一 step 可补入等待任务。这里没有使用 serving 框架，active set、remaining token 和完成回执都由状态机显式维护。

In [3]:
def simulate_continuous(items, batch_capacity):  # 模拟逐 decode step 动态准入的 continuous batching
    time = 0  # 初始化连续调度时间轴
    waiting = []  # 初始化到达但尚未获得 KV slot 的队列
    active = []  # 初始化本 step 参与 decode 的活动序列
    completion = {}  # 保存每条请求生成完成的时刻
    timeline = []  # 收集逐 step 准入、生成与回收轨迹
    reserved_slots = 0  # 统计真正活跃序列占用的总 slot-step
    useful_tokens = 0  # 统计实际生成 token 总量
    while len(completion) < len(items):  # 循环直到八条请求全部完成
        waiting.extend(dict(item, remaining=item["new_tokens"]) for item in items if item["arrival"] == time)  # 接收当前 step 新到请求
        admitted = []  # 记录本 step 新获得执行 slot 的请求
        while waiting and len(active) < batch_capacity:  # 持续填补刚释放或初始空闲的 slot
            request = waiting.pop(0)  # 按先到先服务策略取得队首请求
            active.append(request)  # 将请求加入本轮动态 batch
            admitted.append(request["id"])  # 记录准入事件供调度审计
        active_before = [slot["id"] for slot in active]  # 保存生成前活动集合
        reserved_slots += len(active)  # 只累计本 step 真正活跃的序列 slot
        useful_tokens += len(active)  # 每个活动序列都在本 step 生成一个 token
        for slot in active:  # 并行推进当前动态 batch 的所有序列
            slot["remaining"] -= 1  # 消耗当前请求一个生成 token step
            if slot["remaining"] == 0:  # 检查序列是否刚刚完成
                completion[slot["id"]] = time + 1  # 写入完成回执用于端到端延迟计算
        finished = [slot["id"] for slot in active if slot["remaining"] == 0]  # 收集本 step 可以回收 KV slot 的请求
        active = [slot for slot in active if slot["remaining"] > 0]  # 立即从活动集合移除已完成序列
        timeline.append({"step": time, "准入": admitted, "active": active_before, "完成": finished, "等待": [item["id"] for item in waiting]})  # 保存完整动态调度轨迹
        time += 1  # 推进到下一 decode step
    return completion, timeline, reserved_slots, useful_tokens  # 返回逐请求结果与 slot 利用数据
continuous_completion, continuous_timeline, continuous_reserved, continuous_useful = simulate_continuous(requests, capacity)  # 在同一批请求上运行连续调度
print("Continuous batching 前十个 step：")  # 输出核心算法轨迹标题
pprint(continuous_timeline[:10], sort_dicts=False)  # 展示短请求完成后下一 step 如何补入等待任务

Continuous batching 前十个 step：
[{'step': 0,
  '准入': ['Q01', 'Q02', 'Q03'],
  'active': ['Q01', 'Q02', 'Q03'],
  '完成': [],
  '等待': []},
 {'step': 1,
  '准入': [],
  'active': ['Q01', 'Q02', 'Q03'],
  '完成': ['Q02'],
  '等待': ['Q04']},
 {'step': 2,
  '准入': ['Q04'],
  'active': ['Q01', 'Q03', 'Q04'],
  '完成': [],
  '等待': ['Q05']},
 {'step': 3,
  '准入': [],
  'active': ['Q01', 'Q03', 'Q04'],
  '完成': ['Q03'],
  '等待': ['Q05', 'Q06']},
 {'step': 4,
  '准入': ['Q05'],
  'active': ['Q01', 'Q04', 'Q05'],
  '完成': ['Q04', 'Q05'],
  '等待': ['Q06', 'Q07']},
 {'step': 5,
  '准入': ['Q06', 'Q07'],
  'active': ['Q01', 'Q06', 'Q07'],
  '完成': ['Q01'],
  '等待': ['Q08']},
 {'step': 6,
  '准入': ['Q08'],
  'active': ['Q06', 'Q07', 'Q08'],
  '完成': ['Q07'],
  '等待': []},
 {'step': 7, '准入': [], 'active': ['Q06', 'Q08'], '完成': [], '等待': []},
 {'step': 8, '准入': [], 'active': ['Q06', 'Q08'], '完成': [], '等待': []},
 {'step': 9,
  '准入': [],
  'active': ['Q06', 'Q08'],
  '完成': ['Q06', 'Q08'],
  '等待': []}]


## 4. 关键中间量：逐请求完成时刻与 slot 利用率

两种方案生成的 token 总数完全相同，差异来自静态 batch 的已完成 slot 仍被保留。下面计算每条请求的延迟变化以及 `useful_tokens / reserved_slots`。

In [4]:
comparison = []  # 收集同一请求在两种调度器下的完成差异
for item in requests:  # 遍历八条真实请求
    static_latency = static_completion[item["id"]] - item["arrival"]  # 计算静态 batch 端到端延迟
    continuous_latency = continuous_completion[item["id"]] - item["arrival"]  # 计算连续 batch 端到端延迟
    comparison.append({"请求": item["id"], "问题": item["question"], "静态完成": static_completion[item["id"]], "连续完成": continuous_completion[item["id"]], "静态延迟": static_latency, "连续延迟": continuous_latency, "延迟改善": static_latency - continuous_latency})  # 保存逐请求可解释对照
static_utilization = static_useful / static_reserved  # 计算静态 batch 预留 slot 中真正产生 token 的比例
continuous_utilization = continuous_useful / continuous_reserved  # 计算连续 batch 活动 slot 的有效利用率
print("逐请求调度对照：")  # 输出关键中间量标题
pprint(comparison, sort_dicts=False)  # 展示不同长度请求获得的完成时刻改善
print({"静态slot利用率": round(static_utilization, 3), "连续slot利用率": round(continuous_utilization, 3), "生成token总数": static_useful})  # 展示相同工作量下的 slot 浪费差异

逐请求调度对照：
[{'请求': 'Q01',
  '问题': '总结退款政策',
  '静态完成': 6,
  '连续完成': 6,
  '静态延迟': 6,
  '连续延迟': 6,
  '延迟改善': 0},
 {'请求': 'Q02',
  '问题': '订单 A17 到哪了',
  '静态完成': 2,
  '连续完成': 2,
  '静态延迟': 2,
  '连续延迟': 2,
  '延迟改善': 0},
 {'请求': 'Q03',
  '问题': '解释 BM25 参数',
  '静态完成': 4,
  '连续完成': 4,
  '静态延迟': 4,
  '连续延迟': 4,
  '延迟改善': 0},
 {'请求': 'Q04',
  '问题': '生成会议行动项',
  '静态完成': 9,
  '连续完成': 5,
  '静态延迟': 8,
  '连续延迟': 4,
  '延迟改善': 4},
 {'请求': 'Q05',
  '问题': '判断是否需要人工升级',
  '静态完成': 7,
  '连续完成': 5,
  '静态延迟': 5,
  '连续延迟': 3,
  '延迟改善': 2},
 {'请求': 'Q06',
  '问题': '比较 LoRA 与 DoRA',
  '静态完成': 11,
  '连续完成': 10,
  '静态延迟': 8,
  '连续延迟': 7,
  '延迟改善': 1},
 {'请求': 'Q07',
  '问题': '提取发票抬头',
  '静态完成': 13,
  '连续完成': 7,
  '静态延迟': 9,
  '连续延迟': 3,
  '延迟改善': 6},
 {'请求': 'Q08',
  '问题': '给出数据库排障步骤',
  '静态完成': 15,
  '连续完成': 10,
  '静态延迟': 10,
  '连续延迟': 5,
  '延迟改善': 5}]
{'静态slot利用率': 0.659, '连续slot利用率': 1.0, '生成token总数': 27}


## 5. 结果解读：平均延迟改善不意味着每个请求都等量受益

首批最长请求 Q01 的生成路径基本不变，真正受益的是原本被静态批边界挡住的后续短请求。逐样本表保留零改善项，避免只展示平均值；吞吐和延迟应分别观察。

In [5]:
average_static_latency = sum(row["静态延迟"] for row in comparison) / len(comparison)  # 计算静态 batching 的平均端到端延迟
average_continuous_latency = sum(row["连续延迟"] for row in comparison) / len(comparison)  # 计算 continuous batching 的平均端到端延迟
improved_requests = [row["请求"] for row in comparison if row["延迟改善"] > 0]  # 找出真正因动态补位提前完成的请求
unchanged_requests = [row["请求"] for row in comparison if row["延迟改善"] == 0]  # 保留未受益样本防止选择性汇报
print({"静态平均延迟": round(average_static_latency, 3), "连续平均延迟": round(average_continuous_latency, 3), "获得改善的请求": improved_requests, "未改变的请求": unchanged_requests, "静态浪费slot-step": static_reserved - static_useful})  # 汇总结果并明确收益边界

{'静态平均延迟': 6.5, '连续平均延迟': 4.25, '获得改善的请求': ['Q04', 'Q05', 'Q06', 'Q07', 'Q08'], '未改变的请求': ['Q01', 'Q02', 'Q03'], '静态浪费slot-step': 14}


## 6. 失败案例与修正：只看优先级会让老请求长期饥饿

Continuous batching 只解决补位，不自动保证公平。当前时刻 10 若永远选择新到的高优先级请求，等待十个 step 的低优先级长文可能一直进不去；aging 将等待时间加入分数，使它最终越过新请求。

In [6]:
current_time = 10  # 设置观察公平性的当前调度时刻
waiting_jobs = [{"id": "LOW-OLD", "question": "生成长合规报告", "arrival": 0, "priority": 1}, {"id": "HIGH-NEW", "question": "在线支付风控判断", "arrival": 10, "priority": 5}, {"id": "MID", "question": "客服摘要", "arrival": 7, "priority": 3}]  # 构造老低优先级与新高优先级竞争的真实队列
priority_only = max(waiting_jobs, key=lambda item: item["priority"])  # 复现只按静态优先级选择新请求的饥饿策略
aging_rate = 0.6  # 设置每等待一个 step 获得的公平性加分
aging_scores = {item["id"]: item["priority"] + aging_rate * (current_time - item["arrival"]) for item in waiting_jobs}  # 计算优先级与等待时长共同形成的准入分数
aging_choice = max(waiting_jobs, key=lambda item: aging_scores[item["id"]])  # 用 aging 分数选择已经等待很久的任务
print({"失败_纯优先级选择": priority_only["id"], "各请求aging分数": aging_scores, "修正_aging选择": aging_choice["id"], "LOW-OLD已等待": current_time - waiting_jobs[0]["arrival"]})  # 展示饥饿复现和公平修正

{'失败_纯优先级选择': 'HIGH-NEW', '各请求aging分数': {'LOW-OLD': 7.0, 'HIGH-NEW': 5.0, 'MID': 4.8}, '修正_aging选择': 'LOW-OLD', 'LOW-OLD已等待': 10}


## 7. 生产差距与最小回归检查

真实调度器还要区分 prefill 与 decode 成本，按 KV block 而不是简单请求数做容量控制，并支持 chunked prefill、取消、超时、adapter 分组和抢占。预计生成长度通常未知，应以真实显存水位和历史分布动态准入；公平性要报告最大等待与租户配额。下面的最小断言只验证本实验已展示的同工作量、slot 回收、延迟和 aging。

In [7]:
assert len(requests) >= 6  # 确认真实请求数量满足逐样本系统教学要求
assert static_useful == continuous_useful == sum(item["new_tokens"] for item in requests)  # 确认两种调度器完成完全相同的 token 工作量
assert continuous_reserved < static_reserved  # 确认动态回收减少已完成序列占用的 slot-step
assert continuous_utilization > static_utilization  # 确认 continuous batching 提高有效 slot 利用率
assert average_continuous_latency < average_static_latency  # 确认同请求集合的平均端到端延迟下降
assert priority_only["id"] == "HIGH-NEW" and aging_choice["id"] == "LOW-OLD"  # 确认饥饿失败与 aging 修正均被复现
print("回归检查通过：动态补位、slot 利用率、逐请求延迟与公平性 aging 均已验证。")  # 输出最终验收结论

回归检查通过：动态补位、slot 利用率、逐请求延迟与公平性 aging 均已验证。
